# Mining a Declare model and measuring fitness

This notebook mines a Declare model from a small event log and scores traces against it. The log, `loan_applications.csv`, holds eight cases of a loan application process, one row per event with the columns `case_id`, `activity`, and `timestamp`.

## Load the log

FCFDeclare reads the control flow only, so a log is a list of traces and a trace is the list of its activity names in order. The events of each case are sorted by timestamp, since the rows of a CSV need not be in order.

In [1]:
import csv
from collections import defaultdict

import fcfdeclare as fcf

events = defaultdict(list)
with open('loan_applications.csv', newline='') as file:
    for row in csv.DictReader(file):
        events[row['case_id']].append((row['timestamp'], row['activity']))

cases = {case: [activity for _, activity in sorted(rows)] for case, rows in events.items()}
log = list(cases.values())
for case, trace in cases.items():
    print(case, ' -> '.join(trace))

C1 Register -> Check -> Approve -> Notify -> Close
C2 Register -> Check -> Reject -> Notify -> Close
C3 Register -> Check -> Request Info -> Check -> Approve -> Notify -> Close
C4 Register -> Check -> Approve -> Notify -> Close
C5 Register -> Check -> Reject -> Notify -> Close
C6 Register -> Check -> Request Info -> Check -> Reject -> Notify -> Close
C7 Register -> Check -> Approve -> Notify -> Close
C8 Register -> Check -> Approve -> Close


ISO 8601 timestamps sort correctly as strings. For other formats, parse them with `datetime.strptime` first.

## Mine the model

Every template is a candidate on every activity, or on every ordered pair of distinct activities. A candidate is kept when both its measures reach their thresholds:

- **support**: the fraction of the traces in which its activation occurs and it holds.
- **confidence**: the fraction of the traces in which its activation occurs that it holds in.

A `min_confidence` below 1 tolerates a few deviating cases, such as `C8`, which closes without notifying the applicant.

In [2]:
model = fcf.mine(log, min_support=0.5, min_confidence=0.85, max_cardinality=2)
print(f'{len(model.constraints)} constraints')

179 constraints


Each constraint maps to its measures. The constraints below are binary ones with a confidence under 1, which some case of the log violates.

In [3]:
for constraint, measures in model.constraints.items():
    if constraint.template.is_binary and measures.confidence < 1:
        print(f'{constraint}: support {measures.support:.2f}, confidence {measures.confidence:.2f}')

Choice(Notify, Reject): support 0.88, confidence 0.88
Choice(Notify, Request Info): support 0.88, confidence 0.88
Choice(Reject, Notify): support 0.88, confidence 0.88
Choice(Request Info, Notify): support 0.88, confidence 0.88
Responded Existence(Check, Notify): support 0.88, confidence 0.88
Responded Existence(Close, Notify): support 0.88, confidence 0.88
Responded Existence(Register, Notify): support 0.88, confidence 0.88
Response(Check, Notify): support 0.88, confidence 0.88
Response(Register, Notify): support 0.88, confidence 0.88
Precedence(Notify, Close): support 0.88, confidence 0.88
Chain Precedence(Notify, Close): support 0.88, confidence 0.88
Not Chain Precedence(Approve, Close): support 0.88, confidence 0.88
Alternate Response(Register, Notify): support 0.88, confidence 0.88
Alternate Precedence(Notify, Close): support 0.88, confidence 0.88


## Fitness of the log

A `Checker` scores a batch of traces against the model. A trace in which a constraint's activation does not occur satisfies it vacuously. Two scores come out for each trace:

- `share`: the fraction of the constraints it satisfies, a fitness in $[0, 1]$.
- `full`: 1 when it satisfies every constraint, else 0.

In [4]:
checker = fcf.Checker(model)
result = checker.check(log)
for case, share, full in zip(cases, result.share, result.full, strict=True):
    print(f'{case}: fitness {share:.3f}, conformant {bool(full)}')
print(f'Mean fitness {result.share.mean():.3f}, conformant cases {result.full.mean():.0%}')

C1: fitness 1.000, conformant True
C2: fitness 1.000, conformant True
C3: fitness 1.000, conformant True
C4: fitness 1.000, conformant True
C5: fitness 1.000, conformant True
C6: fitness 1.000, conformant True
C7: fitness 1.000, conformant True
C8: fitness 0.911, conformant False
Mean fitness 0.989, conformant cases 88%


## Fitness of new traces

The model also scores traces it was not mined from. Activities it does not name match no constraint, but still count as events.

In [5]:
new = {
    'conformant': ['Register', 'Check', 'Reject', 'Notify', 'Close'],
    'skips check': ['Register', 'Approve', 'Notify', 'Close'],
    'approve and reject': ['Register', 'Check', 'Approve', 'Reject', 'Notify', 'Close'],
    'unknown activity': ['Register', 'Check', 'Escalate', 'Approve', 'Notify', 'Close'],
}
result = checker.check(new.values())
for name, share in zip(new, result.share, strict=True):
    print(f'{name}: fitness {share:.3f}')

conformant: fitness 1.000
skips check: fitness 0.883
approve and reject: fitness 0.972
unknown activity: fitness 0.994


`holds` gives the satisfaction of each constraint separately, one column per constraint of the model, which shows which rules a trace breaks.

In [6]:
holds = checker.holds(new.values())
constraints = list(model.constraints)
for name, row in zip(new, holds, strict=True):
    violated = [constraints[k] for k in (~row).nonzero()[0]]
    print(f'{name}: {len(violated)} violated')
    for constraint in violated:
        print(f'    {constraint}')

conformant: 0 violated
skips check: 21 violated
    Existence(Check, 1)
    Choice(Check, Reject)
    Choice(Check, Request Info)
    Choice(Reject, Check)
    Choice(Request Info, Check)
    Responded Existence(Approve, Check)
    Responded Existence(Close, Check)
    Responded Existence(Notify, Check)
    Responded Existence(Register, Check)
    Response(Register, Check)
    Precedence(Check, Approve)
    Precedence(Check, Close)
    Precedence(Check, Notify)
    Chain Response(Register, Check)
    Chain Precedence(Check, Approve)
    Not Chain Response(Register, Approve)
    Not Chain Precedence(Register, Approve)
    Alternate Response(Register, Check)
    Alternate Precedence(Check, Approve)
    Alternate Precedence(Check, Close)
    Alternate Precedence(Check, Notify)
approve and reject: 5 violated
    Exclusive Choice(Approve, Reject)
    Exclusive Choice(Reject, Approve)
    Not Responded Existence(Approve, Reject)
    Not Response(Approve, Reject)
    Not Chain Response(Approv

## Save and reload

A model saves to JSON with its settings and measures, and reloads to an equal model.

In [7]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'model.json'
    model.save(path)
    reloaded = fcf.DeclareModel.load(path)
reloaded == model

True